In [10]:
import pandas as pd
import sqlite3

In [11]:
connection = sqlite3.connect('../data/checking-logs.sqlite')

In [12]:
schema = pd.read_sql('PRAGMA table_info(deadlines);', connection)
schema

,cid,name,type,notnull,dflt_value,pk
0,0,index,INTEGER,0,None,0
1,1,labs,TEXT,0,None,0
2,2,deadlines,INTEGER,0,None,0


In [13]:
test = pd.io.sql.read_sql("PRAGMA table_info(test);", connection)
test_first_rows = pd.io.sql.read_sql("SELECT * from test LIMIT 10", connection)
test_first_rows

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


In [21]:
query1 = '''
SELECT
    test.uid
FROM test
JOIN deadlines ON test.labname = deadlines.labs
WHERE test.labname != 'project1'
'''
test_first = pd.io.sql.read_sql(query1, connection)
test_first

,uid
0,user_1
1,user_1
2,user_1
3,user_1
4,user_1
5,user_10
6,user_10
7,user_10
8,user_10
9,user_10


In [26]:
query = '''
SELECT
    test.uid,
    (julianday(test.first_commit_ts) - julianday(datetime(deadlines.deadlines, 'unixepoch'))) * 24 AS diff_hours
FROM test
JOIN deadlines ON test.labname = deadlines.labs
WHERE test.labname != 'project1'
ORDER BY diff_hours
LIMIT 1;
'''

df_min = pd.io.sql.read_sql(query, connection)
df_min


,uid,diff_hours
0,user_30,-202.38473


In [6]:
query = '''
SELECT
    AVG((julianday(datetime(deadlines.deadlines, 'unixepoch')) - julianday(test.first_commit_ts)) * 24) AS avg_hours
FROM test
JOIN deadlines ON test.labname = deadlines.labs
WHERE test.labname != 'project1'
'''
df_max = pd.io.sql.read_sql(query, connection)
df_max

,avg_hours
0,89.687686


In [7]:
query = '''
SELECT
    test.uid AS uid,
    AVG((julianday(datetime(deadlines.deadlines, 'unixepoch')) - julianday(test.first_commit_ts)) * 24) AS avg_diff,
    COUNT(pageviews.uid) AS pageviews
FROM test
JOIN deadlines ON test.labname = deadlines.labs
LEFT JOIN pageviews ON test.uid = pageviews.uid
WHERE test.labname != 'project1'
GROUP BY test.uid;
'''

views_diff = pd.read_sql(query, connection)
views_diff
views_diff.corr(numeric_only=True)

,avg_diff,pageviews
avg_diff,1.000000,0.185042
pageviews,0.185042,1.000000


In [8]:
connection.close()